\title{}
\author{}
\date{}
\makeatletter
\renewcommand{\maketitle}{}
\makeatother

\thispagestyle{empty}

\begin{center}
\vspace*{4cm}

{\LARGE Asset Allocation \& Investment Strategies \\[0.5cm]}

Academic year: 2025--2026\\[1.5cm]

Group 8\\[0.3cm]
Sacha Mimoun\\
Isabelle Chuah\\
Victor Lotigie\\
Evelyn Wang\\
Bolun Tian\\[1.5cm]

\textit{Imperial College Business School}

\end{center}

\newpage

\setcounter{secnumdepth}{0}

\thispagestyle{empty}
\clearpage

\tableofcontents

\newpage

<CENTER>
<p><font size="5"> ASSET ALLOCATION </span></p>
<p><font size="5"> ASSIGNMENT 2 : FAMA-FRENCH FACTORS (PAPER 1993) </font></p>
</p>
</CENTER>

In [121]:
import pandas as pd
import datetime
import os
import statsmodels.api as sm


In [122]:
# Import raw data

path = "raw_data/"
filename_factors = 'F-F_Research_Data_Factors.CSV'
filename_portfolios = '25_Portfolios_5x5.CSV'
filepath = os.path.join(path, filename_factors)
filepath_portfolios = os.path.join(path, filename_portfolios)

In [123]:
# Read F-F factors file - extract only monthly data
df_temp = pd.read_csv(filepath, skiprows=3, header=None)
separator_idx = df_temp[df_temp[0].astype(str).str.contains('Annual', case=False, na=False)].index[0]



In [124]:
# Read only monthly data (stop before annual section)
df_ff_monthly = pd.read_csv(filepath, skiprows=3, nrows=separator_idx-1)
# Clean monthly factors data
df_ff_monthly['date_str'] = df_ff_monthly.iloc[:, 0].astype(str).str.strip()
df_ff_monthly = df_ff_monthly[df_ff_monthly['date_str'].str.match(r'^\d{6}$')]
df_ff_monthly.index = pd.to_datetime(df_ff_monthly['date_str'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_ff_monthly = df_ff_monthly.iloc[:, 1:-1] / 100 # because it was in percentage before
print("Fama-French Monthly Factors:")
print(df_ff_monthly)
print(f"Shape: {df_ff_monthly.shape}")

Fama-French Monthly Factors:
            Mkt-RF     SMB     HML      RF
date_str                                  
1926-07-01  0.0296 -0.0256 -0.0243  0.0022
1926-08-01  0.0264 -0.0117  0.0382  0.0025
1926-09-01  0.0036 -0.0140  0.0013  0.0023
1926-10-01 -0.0324 -0.0009  0.0070  0.0032
1926-11-01  0.0253 -0.0010 -0.0051  0.0031
...            ...     ...     ...     ...
2023-08-01 -0.0239 -0.0316 -0.0106  0.0045
2023-09-01 -0.0524 -0.0251  0.0152  0.0043
2023-10-01 -0.0319 -0.0387  0.0019  0.0047
2023-11-01  0.0884 -0.0002  0.0164  0.0044
2023-12-01  0.0485  0.0635  0.0494  0.0043

[1170 rows x 4 columns]
Shape: (1170, 4)


Let's do the same for the 25 portfolios

In [125]:
# Read 25 Portfolios file - extract only monthly datasets
with open(filepath_portfolios, 'r') as f:
    lines = f.readlines()

# Find indices for monthly datasets
vw_idx = next(i for i, line in enumerate(lines) if 'Average Value Weighted Returns -- Monthly' in line)
ew_idx = next(i for i, line in enumerate(lines) if 'Average Equal Weighted Returns -- Monthly' in line)

print(f"VW Monthly: line {vw_idx}, EW Monthly: line {ew_idx}")

VW Monthly: line 14, EW Monthly: line 1188


In [126]:
# Extract monthly datasets
df_port_vw = pd.read_csv(filepath_portfolios, skiprows=vw_idx+1, nrows=ew_idx-vw_idx-3)
df_port_vw['Date'] = df_port_vw.iloc[:, 0].astype(str).str.strip()
df_port_vw = df_port_vw[df_port_vw['Date'].str.match(r'^\d{6}$', na=False)]
df_port_vw.index = pd.to_datetime(df_port_vw['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_vw = df_port_vw.iloc[:, 1:-1] / 100

df_port_ew = pd.read_csv(filepath_portfolios, skiprows=ew_idx+1, nrows=ew_idx-vw_idx-3)
df_port_ew['Date'] = df_port_ew.iloc[:, 0].astype(str).str.strip()
df_port_ew = df_port_ew[df_port_ew['Date'].str.match(r'^\d{6}$', na=False)]
df_port_ew.index = pd.to_datetime(df_port_ew['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_ew = df_port_ew.iloc[:, 1:-1] / 100 # it was again in percentage

print(f"\nVW Monthly: {df_port_vw.shape}")
print(df_port_vw)

print(f"EW Monthly: {df_port_ew.shape}")

#print(df_port_ew)


VW Monthly: (1170, 25)
            SMALL LoBM   ME1 BM2   ME1 BM3   ME1 BM4  SMALL HiBM   ME2 BM1  \
Date                                                                         
1926-07-01    0.058248 -0.017006  0.004875 -0.014580    0.020534  0.012077   
1926-08-01   -0.020206 -0.080282  0.013796  0.014606    0.083968  0.023618   
1926-09-01   -0.048291 -0.026154 -0.043417 -0.032729    0.008649 -0.026540   
1926-10-01   -0.093729 -0.035519 -0.034948  0.034413   -0.025476 -0.028069   
1926-11-01    0.055888  0.041877  0.024623 -0.044494    0.005362  0.031033   
...                ...       ...       ...       ...         ...       ...   
2023-08-01   -0.122376 -0.076627 -0.105103 -0.056405   -0.073431 -0.069345   
2023-09-01   -0.082275 -0.076201 -0.065856 -0.058185   -0.060216 -0.088904   
2023-10-01   -0.102240 -0.089443 -0.079196 -0.063774   -0.076783 -0.102040   
2023-11-01    0.057861  0.080737  0.107319  0.085040    0.070945  0.109883   
2023-12-01    0.153216  0.160383  0.1486

In [127]:
# Compute excess returns for the 25 portfolios
df_port_vw_excess = df_port_vw.sub(df_ff_monthly['RF'], axis=0)
df_port_ew_excess = df_port_ew.sub(df_ff_monthly['RF'], axis=0)

In [128]:
# Prepare results storage
results = []

mkt_rf = df_ff_monthly['Mkt-RF']

for col in df_port_vw_excess.columns:
    y = df_port_vw_excess[col]
    X = sm.add_constant(mkt_rf)
    
    model = sm.OLS(y, X).fit()
    
    results.append({
        'portfolio': col,
        'alpha': model.params['const'],
        'beta': model.params['Mkt-RF'],
        'alpha_tstat': model.tvalues['const'],
        'beta_tstat': model.tvalues['Mkt-RF'],
        'r_squared': model.rsquared
    })

df_results = pd.DataFrame(results)

print("\n Regression Results :")
print(df_results.to_string(index=False))



 Regression Results :
 portfolio     alpha     beta  alpha_tstat  beta_tstat  r_squared
SMALL LoBM -0.005286 1.606437    -2.138860   35.001418   0.511930
   ME1 BM2 -0.002653 1.392951    -1.443571   40.816448   0.587859
   ME1 BM3  0.000600 1.392947     0.377618   47.246427   0.656493
   ME1 BM4  0.002934 1.259552     2.071882   47.899528   0.662658
SMALL HiBM  0.004219 1.359938     2.509640   43.565664   0.619044
   ME2 BM1 -0.002274 1.268873    -1.833529   55.105964   0.722213
   ME2 BM2  0.001075 1.226088     1.009736   62.041944   0.767201
   ME2 BM3  0.001559 1.196738     1.578868   65.268790   0.784820
   ME2 BM4  0.002203 1.205568     2.078456   61.248719   0.762573
   ME2 BM5  0.003158 1.375052     2.318735   54.365067   0.716750
   ME3 BM1 -0.001297 1.248609    -1.370642   71.076510   0.812215
   ME3 BM2  0.001417 1.126204     2.030826   86.915724   0.866091
   ME3 BM3  0.001608 1.118466     2.181422   81.720863   0.851140
   ME3 BM4  0.002214 1.173756     2.459152   70.20715